# Notebook 11 — LaTeX and Scientific Tools

**Module:** 18 — Scientific Writing and Open Science  
**Tier:** 2 — Working competence  
**Notebook:** 11 of 12  
**Time estimate:** 90 minutes

> LaTeX is the standard document format for computational biology papers,
> theses, and PhD applications. This notebook covers the minimum viable
> LaTeX workflow: document structure, equations, figures, references,
> and the Overleaf / local toolchain.

---
## Step 1 — Why LaTeX for Scientific Writing

LaTeX separates content from formatting. You write the text; the compiler
places everything correctly. Benefits:
- Equations typeset correctly (critical for methods sections)
- References managed automatically (BibTeX / biber)
- Figure placement is determined by the journal template
- Diffs are meaningful (plain text files, trackable in git)
- All major journals provide `.cls` / `.sty` template files

**When not to use LaTeX:** single-author documents with no equations and no
bibliography (use Markdown). Short emails and slides (use Beamer or plain
HTML). LaTeX has a steep initial cost — only pay it for documents where
its advantages justify the learning curve.

---
## Step 2 — Minimum Viable LaTeX Document

```latex
\documentclass[12pt]{article}
\usepackage{amsmath}        % equations
\usepackage{graphicx}       % figures
\usepackage[colorlinks=true, linkcolor=blue]{hyperref}  % hyperlinks
\usepackage[numbers]{natbib} % references

\title{DeepBind-v2: Transcription Factor Binding Prediction}
\author{Vinoth A.\thanks{github.com/Vinoth-ai-20} \\
        \small Department of Computational Biology}
\date{\today}

\begin{document}
\maketitle
\begin{abstract}
  [6-sentence abstract here]
\end{abstract}

\section{Introduction}
[Four-paragraph introduction here]

\section{Methods}
\subsection{Dataset}
[...]

\section{Results}
\begin{figure}[ht]
  \centering
  \includegraphics[width=0.9\columnwidth]{pub_quality_figures.png}
  \caption{AUROC comparison. (A) Paired comparison across 40 TFs.
            (B) Learning curves (n=5 replicates; shaded = $\pm$1 SD).}
  \label{fig:auroc}
\end{figure}
As shown in Figure~\ref{fig:auroc}A, DeepBind-v2 achieves
AUROC $0.924 \pm 0.008$.

\section{Discussion}
[...]

\bibliographystyle{abbrvnat}
\bibliography{references}

\end{document}
```

---
## Step 3 — Equations in LaTeX

**Inline:** `$f(x) = \sigma(Wx + b)$` → $f(x) = \sigma(Wx + b)$

**Display (numbered):**
```latex
\begin{equation}
  \text{AUROC} = \int_0^1 \text{TPR}(\text{FPR}^{-1}(t))\, dt
  \label{eq:auroc}
\end{equation}
```

**Key packages:**
- `amsmath`: `align`, `equation`, `split`, `cases`
- `amssymb`: `\mathbb{R}`, `\mathcal{L}`, `\in`, `\sum`, `\prod`
- `bm`: bold math `\bm{x}` for vectors

**Common patterns in computational biology:**
```latex
% Log-likelihood
\mathcal{L}(\theta) = \sum_{i=1}^N \log p(x_i \mid \theta)

% SIR model
\frac{dS}{dt} = -\frac{\beta S I}{N}, \quad
\frac{dI}{dt} =  \frac{\beta S I}{N} - \gamma I, \quad
\frac{dR}{dt} =  \gamma I

% Softmax
p_k = \frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}}
```

---
## Step 4 — BibTeX and References

**A BibTeX entry:**
```bibtex
@article{alipanahi2015predicting,
  title={Predicting the sequence specificities of DNA and RNA-binding proteins
         by deep learning},
  author={Alipanahi, Babak and others},
  journal={Nature Biotechnology},
  volume={33},
  number={8},
  pages={831--838},
  year={2015},
  doi={10.1038/nbt.3300}
}
```

**Cite in text:** `\citep{alipanahi2015predicting}` → (Alipanahi et al., 2015)

**Where to get BibTeX entries:** Google Scholar → " " icon → BibTeX;
doi2bib.org; Zotero (recommended — browser plugin auto-imports).

**Zotero workflow:** browser plugin → add paper to library → export as BibTeX
→ `references.bib` in the same folder as your `.tex` file.

---
## Step 5 — Toolchain Options

| Tool | Platform | Pros | Cons |
|------|---------|------|------|
| **Overleaf** | Browser | Collaborative; no install; live preview | Slow for large files; requires internet |
| **TeX Live (local)** | All OS | Fast; works offline; full control | Install size ~4 GB |
| **MiKTeX (Windows)** | Windows | On-demand package install | Slower than TeX Live |
| **pandoc** | CLI | Markdown → LaTeX → PDF pipeline | Limited LaTeX feature support |

**Recommended for this programme:**
- Day-to-day: Overleaf (free tier is sufficient for single-author papers)
- For the curriculum assignments (A01, A02): local TeX Live so the
  compiled PDF can be committed to the repository

In [ ]:
import subprocess
import os
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---- LaTeX document generation and compilation ----
# This cell generates a minimal LaTeX document and attempts to compile it
# if pdflatex is available. If not, it shows the source.

MINIMAL_TEX = r"""
\documentclass[11pt]{article}
\usepackage{amsmath, amssymb}
\usepackage[margin=2cm]{geometry}
\usepackage[colorlinks]{hyperref}
\title{\textbf{Module 18 NB11: LaTeX Example Document}}
\author{Vinoth A. (\texttt{vinoth.ac.in@gmail.com})}
\date{\today}

\begin{document}
\maketitle

\begin{abstract}
This document demonstrates minimum viable LaTeX for a computational biology paper.
It includes an equation environment, a bibliography reference, and a figure placeholder.
\end{abstract}

\section{Introduction}
Identifying transcription factor binding sites is central to understanding gene regulation.
Existing PWM methods treat positions as independent, missing epistatic interactions~\cite{alipanahi2015predicting}.

\section{Methods}

\subsection{Model}
Let $\mathbf{x} \in \{A,C,G,T\}^L$ be a DNA sequence of length $L$.
The model computes:
\begin{equation}
    \hat{y} = \sigma\!\left(\mathbf{w}^\top \phi(\mathbf{x}) + b\right)
    \label{eq:model}
\end{equation}
where $\phi(\mathbf{x})$ denotes the convolutional feature map and $\sigma$ the sigmoid function.

\subsection{Loss}
Binary cross-entropy loss:
\begin{equation}
    \mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N}
    \bigl[y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\bigr]
\end{equation}

\section{Results}
DeepBind-v2 achieves $\text{AUROC} = 0.924 \pm 0.008$ across 50 TFs,
outperforming the PWM baseline by 12.3 percentage points ($p < 0.001$,
Wilcoxon signed-rank, $n=50$).

\section*{Data and Code Availability}
Code: \url{https://github.com/Vinoth-ai-20/computational-biology}
Data: ENCODE accession ENCSR000AKF.

\begin{thebibliography}{1}
\bibitem{alipanahi2015predicting}
Alipanahi, B. et al. (2015). Predicting the sequence specificities of DNA- and
RNA-binding proteins by deep learning. \textit{Nature Biotechnology}, 33(8), 831--838.
\end{thebibliography}

\end{document}
"""

print('=== Generated LaTeX document ===' )
print(MINIMAL_TEX[:1200] + '\n...(full source in mini_projects/)')

# Save to a temp file so the source is available
with open('example_paper.tex', 'w') as f:
    f.write(MINIMAL_TEX)
print('\nLaTeX source written to: example_paper.tex')

# Attempt compilation (pdflatex must be on PATH)
try:
    result = subprocess.run(
        ['pdflatex', '-interaction=nonstopmode', 'example_paper.tex'],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0:
        print('pdflatex compilation: SUCCESS → example_paper.pdf')
    else:
        print('pdflatex compilation: FAILED (pdflatex not installed or error)')
        print('Install TeX Live or use Overleaf to compile.')
except (FileNotFoundError, subprocess.TimeoutExpired):
    print('pdflatex not found. Install TeX Live locally or use Overleaf.')
    print('The .tex source is ready at example_paper.tex')

In [ ]:
# ---- LaTeX equation catalogue for computational biology ----
# These render as strings only — for actual rendering, use Overleaf or local LaTeX.

EQ_CATALOGUE = {
    'SIR model (ODEs)': r'\frac{dS}{dt} = -\frac{\beta S I}{N}; \quad \frac{dI}{dt} = \frac{\beta S I}{N} - \gamma I',
    'Log-likelihood': r'\mathcal{L}(\theta) = \sum_{i=1}^N \log p(x_i \mid \theta)',
    'Binary cross-entropy': r'\mathcal{L} = -\frac{1}{N}\sum_i [y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)]',
    'Softmax': r'p_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)}',
    'AUROC (integral)': r'\text{AUROC} = \int_0^1 \text{TPR}(\text{FPR}^{-1}(t))\, dt',
    'Fisher information': r'I(\theta) = -\mathbb{E}\!\left[\frac{\partial^2 \log p(X;\theta)}{\partial \theta^2}\right]',
    'Bayes theorem': r'P(\theta \mid x) = \frac{P(x \mid \theta)\, P(\theta)}{P(x)}',
    'Gillespie propensity': r'a_0 = \sum_j a_j(x); \quad \tau = -\ln(u_1) / a_0',
    'Wright-Fisher allele freq': r'p_{t+1} \sim \text{Bin}(2N,\, p_t^\text{sel}) / 2N',
}

print('=== LaTeX equation catalogue for computational biology ===\n')
for name, eq in EQ_CATALOGUE.items():
    print(f'{name}:')
    print(f'  ${eq}$')
    print()

In [ ]:
# Visualization: LaTeX toolchain diagram + equation preview
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Panel A: LaTeX workflow diagram
ax = axes[0]
ax.axis('off')
workflow = [
    ('Write (.tex)', '#4e79a7', 0.88, 'Write in any text editor or Overleaf'),
    ('Add figures (.png/.pdf)', '#59a14f', 0.72, 'Save figures at ≥300 DPI from matplotlib'),
    ('Manage refs (.bib)', '#f28e2b', 0.56, 'Zotero → export BibTeX'),
    ('Compile (pdflatex + bibtex)', '#e15759', 0.40, 'Run: pdflatex → bibtex → pdflatex × 2'),
    ('Output (.pdf)', '#76b7b2', 0.24, 'Submit to journal, share with supervisors'),
]
for (label, color, y, desc) in workflow:
    ax.add_patch(mpatches.FancyBboxPatch((0.05, y-0.06), 0.9, 0.11,
                                           boxstyle='round,pad=0.02', facecolor=color,
                                           alpha=0.3, transform=ax.transAxes))
    ax.text(0.5, y + 0.01, label, transform=ax.transAxes, fontsize=10,
              fontweight='bold', ha='center', va='center')
    ax.text(0.5, y - 0.03, desc, transform=ax.transAxes, fontsize=8,
              ha='center', va='center', color='#444')
    if y > 0.24:
        ax.annotate('', xy=(0.5, y-0.06), xytext=(0.5, y-0.10),
                      xycoords='axes fraction', textcoords='axes fraction',
                      arrowprops=dict(arrowstyle='->', color='grey', lw=2))
ax.set_title('A. LaTeX compilation workflow')

# Panel B: Equation showcase (using matplotlib mathtext)
ax = axes[1]
ax.axis('off')
equations = [
    (r'$\mathcal{L}(\theta) = \sum_{i=1}^N \log p(x_i \mid \theta)$', 'Log-likelihood'),
    (r'$\frac{dS}{dt} = -\frac{\beta S I}{N}$', 'SIR model (S equation)'),
    (r'$p_k = \frac{\exp(z_k)}{\sum_j \exp(z_j)}$', 'Softmax'),
    (r'$P(\theta \mid x) = \frac{P(x\mid\theta)P(\theta)}{P(x)}$', "Bayes' theorem"),
    (r'$\hat{y} = \sigma(\mathbf{w}^\top \phi(\mathbf{x}) + b)$', 'CNN prediction'),
]
for i, (eq, label) in enumerate(equations):
    y = 0.90 - i * 0.17
    ax.text(0.5, y, eq, transform=ax.transAxes, fontsize=12, ha='center', va='center')
    ax.text(0.5, y - 0.06, f'({label})', transform=ax.transAxes, fontsize=8,
              ha='center', va='center', color='grey')
ax.set_title('B. Key equations in LaTeX mathtext\n(renders in matplotlib and Overleaf)')

plt.suptitle('Module 18 NB11: LaTeX and Scientific Tools', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('latex_and_tools.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

See E11 in `exercises/README.md`:
1. Copy the `MINIMAL_TEX` source above into Overleaf and compile it.
   Screenshot the PDF output.
2. Add the SIR model equations from the catalogue in a new `\section{Methods}`
   subsection. Compile again.
3. Write a BibTeX entry for the Wilkinson et al. 2016 FAIR data paper.

---
## Step 10 — Quiz

1. What LaTeX package is needed for the `align` equation environment?
   Write the LaTeX for a two-line aligned system of ODEs.
2. What is the difference between `\cite{}` and `\citep{}`?
   Which should you use in a parenthetical citation?
3. Write the LaTeX BibTeX entry for: "Alipanahi et al., 2015, Nature Biotechnology,
   vol 33, pages 831–838, DOI 10.1038/nbt.3300."
4. What does running `pdflatex → bibtex → pdflatex → pdflatex` accomplish?
   Why does it need to run four times?

---
## Step 12 — Reflection

> *[Compile `example_paper.tex` in Overleaf. Then add the binary cross-entropy
> loss equation from the catalogue as a numbered equation environment in the
> Methods section. Reference it in the text as Equation~\ref{eq:loss}.
> What was the hardest part to get right?]*

---
*Next: `12_mini_project_and_assessment.ipynb`*